In [1]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
import time

In [9]:
def load_dataset(data_path):
    train_images = []
    test_images = []
    test_masks = []
    test_labels = []

    train_dir = os.path.join(data_path, "train", "good")
    for fname in sorted(os.listdir(train_dir)):
        path = os.path.join(train_dir, fname)
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        train_images.append(img)

    test_dir = os.path.join(data_path, "test")
    gt_dir = os.path.join(data_path, "ground_truth")

    for defect_type in sorted(os.listdir(test_dir)):
        defect_path = os.path.join(test_dir, defect_type)

        for fname in sorted(os.listdir(defect_path)):
            img_path = os.path.join(defect_path, fname)

            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            test_images.append(img)
            test_labels.append(defect_type)

            if defect_type == "good":
                mask = np.zeros(img.shape[:2], dtype=np.uint8)

            else:
                gt_subdir = os.path.join(gt_dir, defect_type)

                mask_name = fname.replace(".png", "_mask.png")
                mask_path = os.path.join(gt_subdir, mask_name)

                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                mask = (mask > 0).astype(np.uint8) * 255

            test_masks.append(mask)

    return train_images, test_images, test_masks, test_labels

In [5]:
def compute_iou(pred_mask, gt_mask):
    pred = pred_mask > 0
    gt = gt_mask > 0

    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()

    if union == 0:
        return 1.0

    return intersection / union

In [6]:
def evaluate_model(predict_fn, test_images, test_masks, test_labels, max_images=None):
    ious = []
    per_class = {}

    for i, (img, gt_mask, label) in enumerate(zip(test_images, test_masks, test_labels)):
        if max_images is not None and i >= max_images:
            break

        pred_mask = predict_fn(img)

        iou = compute_iou(pred_mask, gt_mask)
        ious.append(iou)

        if label not in per_class:
            per_class[label] = []
        per_class[label].append(iou)

        print(f"[{i}] {label} IoU: {iou:.4f}")

    # --- summary ---
    mean_iou = np.mean(ious)

    print("\n=== RESULTS ===")
    print(f"Mean IoU: {mean_iou:.4f}")

    print("\nPer-class IoU:")
    for cls, values in per_class.items():
        print(f"{cls}: {np.mean(values):.4f}")

    return mean_iou, per_class

In [7]:
def predict(image):
    """Simple thresholding segmentation model.

    Args:
        image: numpy array of shape (H, W, 3), uint8 RGB image.

    Returns:
        Binary mask as numpy array of shape (H, W), uint8 with values 0 or 255.
    """
    gray = np.mean(image, axis=2)
    mask = (gray > 128).astype(np.uint8) * 255
    return mask

In [13]:
data_path = "data"

train_imgs, test_imgs, test_masks, test_labels = load_dataset(data_path)

evaluate_model(predict, test_imgs, test_masks, test_labels)

print()

[0] bent_wire IoU: 0.0104
[1] bent_wire IoU: 0.0115
[2] bent_wire IoU: 0.0194
[3] bent_wire IoU: 0.0090
[4] bent_wire IoU: 0.0261
[5] bent_wire IoU: 0.0351
[6] bent_wire IoU: 0.0164
[7] bent_wire IoU: 0.0080
[8] bent_wire IoU: 0.0058
[9] bent_wire IoU: 0.0199
[10] bent_wire IoU: 0.0254
[11] bent_wire IoU: 0.0676
[12] cable_swap IoU: 0.0124
[13] cable_swap IoU: 0.0214
[14] cable_swap IoU: 0.0309
[15] cable_swap IoU: 0.0147
[16] cable_swap IoU: 0.0008
[17] cable_swap IoU: 0.0477
[18] cable_swap IoU: 0.0140
[19] cable_swap IoU: 0.0000
[20] cable_swap IoU: 0.0132
[21] cable_swap IoU: 0.0177
[22] cable_swap IoU: 0.0447
[23] combined IoU: 0.0219
[24] combined IoU: 0.0163
[25] combined IoU: 0.0177
[26] combined IoU: 0.0178
[27] combined IoU: 0.0032
[28] combined IoU: 0.0048
[29] combined IoU: 0.0000
[30] combined IoU: 0.0106
[31] combined IoU: 0.0131
[32] combined IoU: 0.0013
[33] cut_inner_insulation IoU: 0.0000
[34] cut_inner_insulation IoU: 0.0009
[35] cut_inner_insulation IoU: 0.0002
[36]

In [26]:
def visualize_predictions(
    predict_fn,
    test_images,
    test_masks,
    test_labels,
    max_images=10,
    delay=0,
    display_size=256
):
    for i, (img, gt_mask, label) in enumerate(zip(test_images, test_masks, test_labels)):
        if i >= max_images:
            break

        pred_mask = predict_fn(img)

        pred_mask = (pred_mask > 0).astype(np.uint8) * 255
        gt_mask = (gt_mask > 0).astype(np.uint8) * 255

        img = cv2.resize(img, (display_size, display_size))
        gt_mask = cv2.resize(gt_mask, (display_size, display_size), interpolation=cv2.INTER_NEAREST)
        pred_mask = cv2.resize(pred_mask, (display_size, display_size), interpolation=cv2.INTER_NEAREST)

        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        gt_color = np.zeros_like(img_bgr)
        pred_color = np.zeros_like(img_bgr)

        gt_color[:, :, 1] = gt_mask
        pred_color[:, :, 2] = pred_mask

        overlay = cv2.addWeighted(img_bgr, 0.7, gt_color, 0.3, 0)
        overlay = cv2.addWeighted(overlay, 0.7, pred_color, 0.3, 0)

        def put_text(image, text):
            return cv2.putText(
                image.copy(),
                text,
                (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

        img_vis = put_text(img_bgr, f"Input ({label})")
        gt_vis = put_text(gt_color, "GT")
        pred_vis = put_text(pred_color, "Pred")
        overlay_vis = put_text(overlay, "Overlay")

        top = np.hstack([img_vis, gt_vis])
        bottom = np.hstack([pred_vis, overlay_vis])
        grid = np.vstack([top, bottom])

        cv2.imshow("Result", grid)

        key = cv2.waitKey(delay)
        if key == 27:  # ESC
            break

    cv2.destroyAllWindows()

In [27]:
visualize_predictions(
    predict_fn=predict,
    test_images=test_imgs,
    test_masks=test_masks,
    test_labels=test_labels,
    max_images=20,
    delay=500
)